# Header

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
os.makedirs('/content/drive/MyDrive/DPAS/HW2', exist_ok=True)

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from DataSynthesizer.DataDescriber import DataDescriber
from DataSynthesizer.DataGenerator import DataGenerator
from DataSynthesizer.lib.utils import read_json_file
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score
)

# Data Cleaning


In [5]:
# load train.csv
df = pd.read_csv('/content/drive/MyDrive/DPAS/HW2/train.csv')

In [6]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [7]:
# See the value counts for each attributes
for col in df.columns:
    print(f"The value counts for {col} :")
    print(df[col].value_counts())

The value counts for PassengerId :
PassengerId
9280_02    1
0001_01    1
0002_01    1
0003_01    1
0003_02    1
          ..
0008_01    1
0007_01    1
0006_02    1
0006_01    1
0005_01    1
Name: count, Length: 8693, dtype: int64
The value counts for HomePlanet :
HomePlanet
Earth     4602
Europa    2131
Mars      1759
Name: count, dtype: int64
The value counts for CryoSleep :
CryoSleep
False    5439
True     3037
Name: count, dtype: int64
The value counts for Cabin :
Cabin
G/734/S     8
B/11/S      7
F/1411/P    7
B/82/S      7
G/981/S     7
           ..
G/543/S     1
B/106/P     1
G/542/S     1
F/700/P     1
G/559/P     1
Name: count, Length: 6560, dtype: int64
The value counts for Destination :
Destination
TRAPPIST-1e      5915
55 Cancri e      1800
PSO J318.5-22     796
Name: count, dtype: int64
The value counts for Age :
Age
24.0    324
18.0    320
21.0    311
19.0    293
23.0    292
       ... 
75.0      4
79.0      3
78.0      3
76.0      2
77.0      2
Name: count, Length: 80, d

In [8]:
# Drop all the null values
df.dropna(inplace=True)
print("The null values :")
print(df.isnull().sum())

# Change all the bool values to 0 and 1
df['Transported'] = df['Transported'].astype(int)
df['CryoSleep'] = df['CryoSleep'].astype(int)
df['VIP'] = df['VIP'].astype(int)

# Remove all the duplicates
df = df.drop_duplicates()

The null values :
PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64


In [9]:
# parse Cabin to Deck / CabinNum / Side
df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')
df.drop(columns=['Cabin'], inplace=True)

# parse PassengerId to GroupId / MemberId
df[['GroupId', 'MemberId']] = df['PassengerId'].str.split('_', expand=True).astype(int)
df.drop(columns=['PassengerId'], inplace=True)

In [10]:
# Drop name since it has high cardinality
df.drop(columns=['Name'], inplace=True)

# Applying PrivBayes with DataSynthesizer

In [11]:
# Convert all boolean columns to integers (0 or 1) before saving
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

df.to_csv('spaceship_preprocessed.csv', index=False)
input_file = 'spaceship_preprocessed.csv'
df = pd.read_csv(input_file) # Re-read to ensure consistency, though now it will read ints for formerly bool columns
num_rows = len(df)

# epsilon: privacy budget
#   - smaller = more private, less utility
#   - larger  = less private, more utility
epsilons = [0.1, 0.5, 1.0, 5.0, 10.0]

# k = max number of parent nodes in Bayesian Network
k = 2

# Columns with fewer unique values than this
# are treated as categorical
category_threshold = 5

In [12]:
import DataSynthesizer.DataGenerator as dg_module
import inspect

dg_path = inspect.getfile(dg_module)

with open(dg_path, 'r') as f:
    content = f.read()

# Fix the problematic line
old_line = "                parents_instance = list(eval(parents_instance))"
new_line = "                parents_instance = list(eval(parents_instance)) if isinstance(eval(parents_instance), tuple) else [eval(parents_instance)]"

content = content.replace(old_line, new_line)

with open(dg_path, 'w') as f:
    f.write(content)

print("✅ Patched!")

✅ Patched!


In [13]:
# Generate synthetic data for each epsilon

for epsilon in epsilons:
    print(f"\n{'='*50}")
    print(f"Generating synthetic data with epsilon = {epsilon}")
    print(f"{'='*50}")

    desc_file = f'description_eps{epsilon}.json'
    synthetic_file = f'synthetic_eps{epsilon}.csv'

    # Describe (learn + privatize)
    describer = DataDescriber(category_threshold=category_threshold)
    describer.describe_dataset_in_correlated_attribute_mode(
        dataset_file=input_file,
        k=k,
        epsilon=epsilon,
        attribute_to_is_candidate_key=None,
        attribute_to_is_categorical=None)
    describer.save_dataset_description_to_file(desc_file)
    print(f"Description saved to {desc_file}")

    # Generate the synthetic data
    generator = DataGenerator()
    generator.generate_dataset_in_correlated_attribute_mode(
        num_rows, desc_file
    )
    generator.save_synthetic_data(synthetic_file)
    print(f"Synthetic data saved to {synthetic_file}")

    synthetic_df = pd.read_csv(synthetic_file)
    print(f"Sample:\n{synthetic_df.head()}")


Generating synthetic data with epsilon = 0.1
================ Constructing Bayesian Network (BN) ================
Adding ROOT GroupId
Adding attribute MemberId
Adding attribute VRDeck
Adding attribute Destination
Adding attribute FoodCourt
Adding attribute Spa
Adding attribute CabinNum
Adding attribute RoomService
Adding attribute CryoSleep
Adding attribute VIP
Adding attribute Side
Adding attribute Age
Adding attribute Transported
Adding attribute HomePlanet
Adding attribute ShoppingMall
========================== BN constructed ==========================
Description saved to description_eps0.1.json


/usr/local/lib/python3.12/dist-packages/DataSynthesizer/datatypes/StringAttribute.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['n' 'o' 'e' ... 'v' '' '']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  column[~column.isnull()] = column[~column.isnull()].apply(lambda x: utils.generate_random_string(int(x)))


Synthetic data saved to synthetic_eps0.1.csv
Sample:
  HomePlanet  CryoSleep    Destination   Age  VIP  RoomService  FoodCourt  \
0     Europa          1    55 Cancri e  63.0    1       3556.0    19033.0   
1      Earth          0    TRAPPIST-1e  15.0    0        422.0    22832.0   
2       Mars          0    TRAPPIST-1e   6.0    1       7329.0    11824.0   
3       Mars          0  PSO J318.5-22  21.0    1       5506.0    28301.0   
4     Europa          0    55 Cancri e  38.0    0       9293.0    19638.0   

   ShoppingMall      Spa   VRDeck  Transported Deck  CabinNum Side  GroupId  \
0        5554.0  17465.0   1649.0            0    n     421.0    P   5036.0   
1       10010.0    281.0  10609.0            0    o     836.0    S   6319.0   
2        5620.0  11969.0  13593.0            1    e     612.0    S   5379.0   
3        2646.0   5749.0   9293.0            1    y     295.0    S   4875.0   
4        2119.0    544.0  13171.0            1    a    1609.0    S   3791.0   

   Member

/usr/local/lib/python3.12/dist-packages/DataSynthesizer/datatypes/StringAttribute.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['n' 'o' 'e' ... 'i' 'k' 's']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  column[~column.isnull()] = column[~column.isnull()].apply(lambda x: utils.generate_random_string(int(x)))


Synthetic data saved to synthetic_eps0.5.csv
Sample:
  HomePlanet  CryoSleep  Destination   Age  VIP  RoomService  FoodCourt  \
0      Earth          1  TRAPPIST-1e  15.0    1       4052.0     4126.0   
1     Europa          1  TRAPPIST-1e  39.0    1       7366.0    24323.0   
2       Mars          0  55 Cancri e  57.0    0       5345.0    26730.0   
3     Europa          1  TRAPPIST-1e  44.0    1       6994.0    17867.0   
4     Europa          1  TRAPPIST-1e  34.0    1       2845.0    13676.0   

   ShoppingMall     Spa  VRDeck  Transported Deck  CabinNum Side  GroupId  \
0        6779.0  1779.0  5716.0            1    n    1558.0    P   5036.0   
1         207.0   281.0  3491.0            1    o     646.0    P   6319.0   
2        6233.0   765.0  2408.0            1    e     707.0    P   5379.0   
3        3258.0   147.0  3193.0            0    y     485.0    S   4875.0   
4       10696.0   544.0  6053.0            1    a    1230.0    S   3791.0   

   MemberId  
0       6.0  
1    

/usr/local/lib/python3.12/dist-packages/DataSynthesizer/datatypes/StringAttribute.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['n' 'o' 'e' ... 'p' 'u' 'g']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  column[~column.isnull()] = column[~column.isnull()].apply(lambda x: utils.generate_random_string(int(x)))


Synthetic data saved to synthetic_eps1.0.csv
Sample:
  HomePlanet  CryoSleep    Destination   Age  VIP  RoomService  FoodCourt  \
0     Europa          1  PSO J318.5-22  70.0    0       1076.0    27976.0   
1       Mars          0  PSO J318.5-22  15.0    1       9846.0      472.0   
2      Earth          1    TRAPPIST-1e  29.0    0       6337.0    22259.0   
3     Europa          1  PSO J318.5-22  17.0    1       6498.0    25320.0   
4     Europa          1    55 Cancri e  18.0    1       9789.0     9204.0   

   ShoppingMall      Spa   VRDeck  Transported Deck  CabinNum Side  GroupId  \
0        9842.0  17465.0  15884.0            1    n    1368.0    S   5036.0   
1        8172.0  13726.0   6542.0            1    o    1120.0    P   6319.0   
2       11134.0   1886.0   3425.0            1    e    1086.0    S   5379.0   
3       10610.0  20314.0  11327.0            1    y      11.0    P   4875.0   
4        7020.0  17350.0   6053.0            0    a     757.0    P   3791.0   

   Member

/usr/local/lib/python3.12/dist-packages/DataSynthesizer/datatypes/StringAttribute.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['n' 'o' 'e' ... 'r' 'i' 's']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  column[~column.isnull()] = column[~column.isnull()].apply(lambda x: utils.generate_random_string(int(x)))


Synthetic data saved to synthetic_eps5.0.csv
Sample:
  HomePlanet  CryoSleep    Destination   Age  VIP  RoomService  FoodCourt  \
0       Mars          1  PSO J318.5-22  23.0    0       1572.0     1145.0   
1       Mars          1    55 Cancri e  74.0    1       8854.0    24323.0   
2      Earth          0    55 Cancri e   2.0    0        385.0     1389.0   
3      Earth          1  PSO J318.5-22  41.0    0       9474.0     7432.0   
4      Earth          0  PSO J318.5-22  14.0    1       5325.0      260.0   

   ShoppingMall    Spa   VRDeck  Transported Deck  CabinNum Side  GroupId  \
0        9230.0  659.0    632.0            1    n     989.0    P   5036.0   
1         207.0  281.0   5525.0            1    o    1309.0    P   6319.0   
2        3783.0  765.0    374.0            0    e     328.0    P   5379.0   
3         808.0  147.0  19461.0            1    y     295.0    P   4875.0   
4         281.0  544.0  14187.0            0    a     851.0    S   3791.0   

   MemberId  
0      

/usr/local/lib/python3.12/dist-packages/DataSynthesizer/datatypes/StringAttribute.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['n' 'o' 'e' ... 'f' 't' 'w']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  column[~column.isnull()] = column[~column.isnull()].apply(lambda x: utils.generate_random_string(int(x)))


# Training the Model

In [16]:
cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']

In [17]:
# Load the original data as the baseline
real_df = pd.read_csv('spaceship_preprocessed.csv')

# OHE real data
real_df = pd.get_dummies(real_df, columns=cat_cols)

target = 'Transported'
X_real = real_df.drop(columns=[target])
y_real = real_df[target]

scaler = StandardScaler()
X_real_scaled = scaler.fit_transform(X_real)

# We only use X_test / y_test from real data for evaluation
_, X_test, _, y_test = train_test_split(
    X_real_scaled, y_real, test_size=0.2, random_state=42
)


# Train the baseline model
X_train_real, _, y_train_real, _ = train_test_split(
    X_real_scaled, y_real, test_size=0.2, random_state=42
)

model_real = SVC(kernel='rbf', C=1.0, gamma='scale')
model_real.fit(X_train_real, y_train_real)
pred_real = model_real.predict(X_test)

print("=" * 55)
print("BASELINE — Trained on REAL data")
print("=" * 55)
print(f"Accuracy  : {accuracy_score(y_test, pred_real):.4f}")
print(f"Precision : {precision_score(y_test, pred_real):.4f}")
print(f"Recall    : {recall_score(y_test, pred_real):.4f}")
print(f"AUC       : {roc_auc_score(y_test, pred_real):.4f}")

BASELINE — Trained on REAL data
Accuracy  : 0.8185
Precision : 0.8235
Recall    : 0.8161
AUC       : 0.8185


In [18]:
results = []

for epsilon in epsilons:
    synthetic_file = f'synthetic_eps{epsilon}.csv'

    try:
        syn_df = pd.read_csv(synthetic_file)
    except FileNotFoundError:
        print(f"\n⚠️  {synthetic_file} not found, skipping.")
        continue

    # OHE synthetic data
    syn_df = pd.get_dummies(syn_df, columns=cat_cols)

    # Align columns with real data (OHE may differ slightly)
    syn_df = syn_df.reindex(columns=real_df.columns, fill_value=0)

    X_syn = syn_df.drop(columns=[target])
    y_syn = syn_df[target]

    # Scale using the SAME scaler fitted on real data
    X_syn_scaled = scaler.transform(X_syn)

    # Train SVC — exactly same settings as HW1
    model = SVC(kernel='rbf', C=1.0, gamma='scale')
    model.fit(X_syn_scaled, y_syn)
    predictions = model.predict(X_test)

    acc  = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec  = recall_score(y_test, predictions)
    auc  = roc_auc_score(y_test, predictions)

    results.append({
        'Epsilon': epsilon,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'AUC': auc
    })

    print(f"\n{'='*55}")
    print(f"DP Synthetic Data  —  epsilon = {epsilon}")
    print(f"{'='*55}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"AUC       : {auc:.4f}")


DP Synthetic Data  —  epsilon = 0.1
Accuracy  : 0.5136
Precision : 0.5100
Recall    : 0.9895
AUC       : 0.5078

DP Synthetic Data  —  epsilon = 0.5
Accuracy  : 0.4917
Precision : 0.4897
Recall    : 0.1061
AUC       : 0.4964

DP Synthetic Data  —  epsilon = 1.0
Accuracy  : 0.5045
Precision : 0.5053
Recall    : 0.9925
AUC       : 0.4986

DP Synthetic Data  —  epsilon = 5.0
Accuracy  : 0.7224
Precision : 0.8213
Recall    : 0.5770
AUC       : 0.7242

DP Synthetic Data  —  epsilon = 10.0
Accuracy  : 0.7224
Precision : 0.8213
Recall    : 0.5770
AUC       : 0.7242


In [19]:
results_df = pd.DataFrame(results)
print("\n\n===== SUMMARY TABLE =====")
print(results_df.to_string(index=False))



===== SUMMARY TABLE =====
 Epsilon  Accuracy  Precision   Recall      AUC
     0.1  0.513616   0.510015 0.989537 0.507785
     0.5  0.491679   0.489655 0.106129 0.496403
     1.0  0.504539   0.505327 0.992526 0.498560
     5.0  0.722390   0.821277 0.576981 0.724172
    10.0  0.722390   0.821277 0.576981 0.724172
